In [1]:
import sys
import pandas as pd
import pyarrow
import sklearn
import matplotlib
import ipykernel

print("Python utilizado:")
print(sys.executable)
print("\nLas librerías se importaron correctamente.")

Python utilizado:
c:\Users\FABIA\OneDrive\Escritorio\Machine learning\ml-practica1-pesantez\venv\Scripts\python.exe

Las librerías se importaron correctamente.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Localizar el proyecto tanto si ejecutamos desde notebooks como desde la raíz.
raiz_proyecto = Path.cwd()
if raiz_proyecto.name == "notebooks":
    raiz_proyecto = raiz_proyecto.parent

ruta_dataset = raiz_proyecto / "data" / "processed" / "dataset.parquet"

print("Archivo que vamos a cargar:")
print(ruta_dataset)

# Cargar los datos procesados de la Práctica 1.
df = pd.read_parquet(ruta_dataset)

print(f"\nNúmero de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

print("\nNombres de las columnas:")
print(df.columns.tolist())

print("\nPrimeras cinco filas:")
display(df.head())

print("\nTipos de datos:")
print(df.dtypes)

Archivo que vamos a cargar:
c:\Users\FABIA\OneDrive\Escritorio\Machine learning\ml-practica1-pesantez\data\processed\dataset.parquet

Número de filas: 5456
Número de columnas: 26

Nombres de las columnas:
['US1', 'US2', 'US3', 'US4', 'US5', 'US6', 'US7', 'US8', 'US9', 'US10', 'US11', 'US12', 'US13', 'US14', 'US15', 'US16', 'US17', 'US18', 'US19', 'US20', 'US21', 'US22', 'US23', 'US24', 'Movimiento', 'Conjunto']

Primeras cinco filas:


,US1,US2,US3,US4,US5,US6,US7,US8,US9,US10,...,US17,US18,US19,US20,US21,US22,US23,US24,Movimiento,Conjunto
0,-0.555806,-0.325067,-0.535488,-0.442497,-0.491462,-0.199362,0.580033,1.188018,0.686723,1.217881,...,-0.156981,0.032680,0.080838,0.606897,0.209977,-0.150837,-0.336249,-0.605744,Move-Forward,train
1,-0.216460,1.578024,2.395991,0.148354,0.046320,-0.155168,-0.255974,-0.269598,0.686723,-0.868642,...,-0.383396,-0.179739,-0.125749,13.189272,0.622970,2.986270,0.045999,-0.177546,Sharp-Right-Turn,train
2,-0.961669,-1.057612,-1.266108,0.512185,0.108342,0.125215,-0.094272,0.286807,-0.341955,-0.192187,...,-0.588679,-0.683007,-0.646707,-0.640613,-0.697216,-0.429942,-0.784580,-1.061358,Slight-Right-Turn,train
3,0.104848,-0.360168,-0.498670,-0.417700,0.911874,0.253867,-0.011129,0.653282,0.323513,1.217881,...,-0.419623,-0.614379,-0.604790,-0.597701,-0.254060,2.986270,0.137998,0.191906,Slight-Right-Turn,train
4,-0.496054,-0.754674,-0.451217,-0.400171,0.911874,-0.553891,0.580033,1.196941,0.686723,1.217881,...,-0.320000,-0.258170,-0.221557,0.196169,0.395592,0.105699,-0.231293,-0.528721,Move-Forward,train



Tipos de datos:
US1           float64
US2           float64
US3           float64
US4           float64
US5           float64
US6           float64
US7           float64
US8           float64
US9           float64
US10          float64
US11          float64
US12          float64
US13          float64
US14          float64
US15          float64
US16          float64
US17          float64
US18          float64
US19          float64
US20          float64
US21          float64
US22          float64
US23          float64
US24          float64
Movimiento        str
Conjunto          str
dtype: object


In [2]:
# Contar cuántos registros hay de cada movimiento.
conteo = df["Movimiento"].value_counts(dropna=False)

distribucion = pd.DataFrame({
    "Cantidad": conteo,
    "Porcentaje": (conteo / len(df) * 100).round(2)
})

print("Distribución de movimientos:")
display(distribucion)

# Revisar si existe una separación previa de los datos.
print("\nValores de la columna Conjunto:")
print(df["Conjunto"].value_counts(dropna=False))

print("\nMovimientos por conjunto:")
display(pd.crosstab(df["Movimiento"], df["Conjunto"]))

# Comprobar si faltan etiquetas.
print("\nValores faltantes en Movimiento:")
print(df["Movimiento"].isna().sum())

Distribución de movimientos:


,Cantidad,Porcentaje
Movimiento,,
Move-Forward,2205,40.41
Sharp-Right-Turn,2097,38.43
Slight-Right-Turn,826,15.14
Slight-Left-Turn,328,6.01



Valores de la columna Conjunto:
Conjunto
train    4364
test     1092
Name: count, dtype: int64

Movimientos por conjunto:


Conjunto,test,train
Movimiento,,
Move-Forward,441,1764
Sharp-Right-Turn,420,1677
Slight-Left-Turn,66,262
Slight-Right-Turn,165,661



Valores faltantes en Movimiento:
0


In [3]:
# Clase minoritaria elegida a partir de las frecuencias observadas.
clase_positiva = "Slight-Left-Turn"

# Objetivo binario: 1 para la clase elegida y 0 para el resto.
y = (df["Movimiento"] == clase_positiva).astype(int)

# Entradas del modelo: únicamente las 24 columnas de sensores.
columnas_sensores = [f"US{i}" for i in range(1, 25)]
X = df[columnas_sensores].copy()

print("Clase positiva (1):", clase_positiva)
print("Clase negativa (0): todos los demás movimientos")
print("\nDimensiones de X:", X.shape)

resumen_binario = y.value_counts().sort_index().to_frame("Cantidad")
resumen_binario["Porcentaje"] = (
    resumen_binario["Cantidad"] / len(y) * 100
).round(2)
resumen_binario.index.name = "Clase binaria"

display(resumen_binario)

print("\nEjemplos de la conversión:")
display(pd.DataFrame({
    "Movimiento original": df["Movimiento"],
    "Objetivo binario": y
}).head(10))

Clase positiva (1): Slight-Left-Turn
Clase negativa (0): todos los demás movimientos

Dimensiones de X: (5456, 24)


,Cantidad,Porcentaje
Clase binaria,,
0,5128,93.99
1,328,6.01



Ejemplos de la conversión:


,Movimiento original,Objetivo binario
0,Move-Forward,0
1,Sharp-Right-Turn,0
2,Slight-Right-Turn,0
3,Slight-Right-Turn,0
4,Move-Forward,0
5,Move-Forward,0
6,Sharp-Right-Turn,0
7,Slight-Right-Turn,0
8,Sharp-Right-Turn,0
9,Move-Forward,0


In [4]:
# Identificar los registros de cada conjunto.
es_train = df["Conjunto"] == "train"
es_test = df["Conjunto"] == "test"

# Separar las entradas (sensores) y las respuestas (clase binaria).
X_train = X.loc[es_train].copy()
X_test = X.loc[es_test].copy()

y_train = y.loc[es_train].copy()
y_test = y.loc[es_test].copy()

# Comprobar tamaños y proporciones.
resumen_particion = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Total": [len(y_train), len(y_test)],
    "Clase 0": [(y_train == 0).sum(), (y_test == 0).sum()],
    "Clase 1": [(y_train == 1).sum(), (y_test == 1).sum()],
    "Porcentaje clase 1": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

display(resumen_particion)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)

,Conjunto,Total,Clase 0,Clase 1,Porcentaje clase 1
0,Entrenamiento,4364,4102,262,6.00
1,Prueba,1092,1026,66,6.04


Dimensiones de X_train: (4364, 24)
Dimensiones de X_test: (1092, 24)


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Crear el modelo.
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Aprender utilizando únicamente los datos de entrenamiento.
modelo_rf.fit(X_train, y_train)

# Predecir las clases de los registros de prueba.
y_pred = modelo_rf.predict(X_test)

print("Entrenamiento completado.")
print("Número de árboles:", modelo_rf.n_estimators)
print("Clases que reconoce el modelo:", modelo_rf.classes_)
print("Cantidad de predicciones:", len(y_pred))

print("\nPrimeras 10 predicciones:")
comparacion = pd.DataFrame({
    "Clase real": y_test.to_numpy(),
    "Clase predicha": y_pred
})
display(comparacion.head(10))

Entrenamiento completado.
Número de árboles: 100
Clases que reconoce el modelo: [0 1]
Cantidad de predicciones: 1092

Primeras 10 predicciones:


,Clase real,Clase predicha
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


In [6]:
from sklearn.metrics import confusion_matrix

# Filas: clase real. Columnas: clase predicha.
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])

tabla_confusion = pd.DataFrame(
    matriz,
    index=["Real 0: otros movimientos", "Real 1: giro leve izquierda"],
    columns=["Predicción 0", "Predicción 1"]
)

display(tabla_confusion)

# Extraer los cuatro tipos de resultados.
tn, fp, fn, tp = matriz.ravel()

print(f"Verdaderos negativos (TN): {tn}")
print(f"Falsos positivos (FP): {fp}")
print(f"Falsos negativos (FN): {fn}")
print(f"Verdaderos positivos (TP): {tp}")

print(f"\nTotal de registros evaluados: {matriz.sum()}")
print(f"Casos positivos reales: {fn + tp}")

,Predicción 0,Predicción 1
Real 0: otros movimientos,1025,1
Real 1: giro leve izquierda,3,63


Verdaderos negativos (TN): 1025
Falsos positivos (FP): 1
Falsos negativos (FN): 3
Verdaderos positivos (TP): 63

Total de registros evaluados: 1092
Casos positivos reales: 66


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, pos_label=1, zero_division=0
)
recall = recall_score(
    y_test, y_pred, pos_label=1, zero_division=0
)
f1 = f1_score(
    y_test, y_pred, pos_label=1, zero_division=0
)

print(f"Accuracy:  {accuracy:.4f} ({accuracy:.2%})")
print(f"Precision: {precision:.4f} ({precision:.2%})")
print(f"Recall:    {recall:.4f} ({recall:.2%})")
print(f"F1:        {f1:.4f}")

print("\nReporte de clasificación:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Otros movimientos", "Giro leve izquierda"],
    digits=4,
    zero_division=0
))

Accuracy:  0.9963 (99.63%)
Precision: 0.9844 (98.44%)
Recall:    0.9545 (95.45%)
F1:        0.9692

Reporte de clasificación:
                     precision    recall  f1-score   support

  Otros movimientos     0.9971    0.9990    0.9981      1026
Giro leve izquierda     0.9844    0.9545    0.9692        66

           accuracy                         0.9963      1092
          macro avg     0.9907    0.9768    0.9836      1092
       weighted avg     0.9963    0.9963    0.9963      1092



In [8]:
import numpy as np

# Clasificador trivial: siempre predice la clase 0.
y_pred_baseline = np.zeros(len(y_test), dtype=int)

# Evaluar ambos clasificadores sobre los mismos registros de prueba.
comparacion_modelos = pd.DataFrame({
    "Modelo": ["Random Forest", "Siempre 0"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_baseline)
    ],
    "Precision": [
        precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        precision_score(y_test, y_pred_baseline, pos_label=1, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        recall_score(y_test, y_pred_baseline, pos_label=1, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        f1_score(y_test, y_pred_baseline, pos_label=1, zero_division=0)
    ]
})

display(comparacion_modelos.round(4))

# Mostrar los aciertos y errores del clasificador trivial.
matriz_baseline = confusion_matrix(
    y_test, y_pred_baseline, labels=[0, 1]
)

print("\nMatriz de confusión del clasificador Siempre 0:")
display(pd.DataFrame(
    matriz_baseline,
    index=["Real 0: otros movimientos", "Real 1: giro leve izquierda"],
    columns=["Predicción 0", "Predicción 1"]
))

,Modelo,Accuracy,Precision,Recall,F1
0,Random Forest,0.9963,0.9844,0.9545,0.9692
1,Siempre 0,0.9396,0.0000,0.0000,0.0000



Matriz de confusión del clasificador Siempre 0:


,Predicción 0,Predicción 1
Real 0: otros movimientos,1026,0
Real 1: giro leve izquierda,66,0


El clasificador “Siempre 0” alcanzó un accuracy del 93,96 %, pero su recall y F1 para la clase minoritaria fueron 0, porque no detectó ningún giro leve a la izquierda.

Random Forest obtuvo un accuracy del 99,63 %, precision del 98,44 %, recall del 95,45 % y F1 de 0,9692; detectó 63 de los 66 positivos, con 1 falso positivo y 3 falsos negativos.

En este conjunto de prueba, la comparación demuestra que el accuracy por sí solo puede ocultar fallos en la clase minoritaria; es necesario interpretarlo junto con precision, recall, F1 y la matriz de confusión.